In [1]:
import pandas as pd
from pathlib import Path

In [4]:
data_path = Path(r'../Initial_data')
recall_dict_path = Path(r'../Results/Recall_dict')
trn_data_path = data_path / 'train_click_log.csv'
tst_data_path = data_path / 'testA_click_log.csv'
item_emb_path = data_path / 'articles_emb.csv'
item_info_path = data_path / 'articles.csv'
trn_data = pd.read_csv(trn_data_path)
tst_data = pd.read_csv(tst_data_path)
item_raw_emb = pd.read_csv(item_emb_path)
item_info = pd.read_csv(item_info_path)
item_info = item_info.rename(columns={'article_id': 'click_article_id'})
all_data = pd.concat([trn_data,tst_data],axis=0) #全量数据，用于生成相似性矩阵

df_filter = all_data.groupby('user_id').filter(lambda x:len(x)>1).reset_index(drop=True)
df_filter = df_filter.sort_values(by=['user_id','click_timestamp'],ascending=True)
trn_last_df = df_filter.groupby('user_id').tail(1).reset_index(drop=True)

In [9]:
article_click_counts = all_data.groupby('click_article_id').size().reset_index(name='click_count')

# 用全量文章表做基准，未被点击的文章点击次数补 0
all_article_ids = item_info[['click_article_id']].drop_duplicates()
article_click_counts_full = all_article_ids.merge(
    article_click_counts,
    on='click_article_id',
    how='left'
 )
article_click_counts_full['click_count'] = article_click_counts_full['click_count'].fillna(0).astype(int)

# 编写函数，统计被点击次数小于某个给定值的文章占比（基于全量文章）
def calculate_click_percentage(article_click_counts_full, threshold):
    # 统计被点击次数小于 threshold 的文章数量
    less_than_threshold = article_click_counts_full[article_click_counts_full['click_count'] < threshold].shape[0]

    # 总文章数量（全量文章）
    total_articles = article_click_counts_full.shape[0]

    # 计算占比
    percentage = (less_than_threshold / total_articles) * 100

    return percentage

# 批量计算 threshold=1~10 的占比
threshold_results = pd.DataFrame({
    'threshold': list(range(1, 11)),
    'percentage': [
        calculate_click_percentage(article_click_counts_full, threshold)
        for threshold in range(1, 11)
    ]
})
threshold_results['percentage'] = threshold_results['percentage'].round(2)

print('被点击次数小于各阈值(threshold=1~10)的文章占比：')
print(threshold_results.to_string(index=False))

被点击次数小于各阈值(threshold=1~10)的文章占比：
 threshold  percentage
         1       90.28
         2       95.43
         3       96.44
         4       96.91
         5       97.23
         6       97.48
         7       97.66
         8       97.81
         9       97.95
        10       98.06


In [10]:
# 计算每个用户的点击次数
user_click_counts = all_data.groupby('user_id').size().reset_index(name='click_count')

# 编写函数，统计用户点击次数小于某个给定值的用户占比
def calculate_user_click_percentage(user_click_counts, threshold):
    # 统计点击次数小于 threshold 的用户数量
    less_than_threshold = user_click_counts[user_click_counts['click_count'] < threshold].shape[0]
    
    # 总用户数量
    total_users = user_click_counts.shape[0]
    
    # 计算占比
    percentage = (less_than_threshold / total_users) * 100
    
    return percentage

# 示例：计算点击次数小于 5 的用户占比
thresholds = [1,2,5,10,20,30]
for threshold in thresholds:
    percentage = calculate_user_click_percentage(user_click_counts, threshold)
    print(f"点击次数小于 {threshold} 的用户占比为: {percentage:.2f}%")

点击次数小于 1 的用户占比为: 0.00%
点击次数小于 2 的用户占比为: 3.81%
点击次数小于 5 的用户占比为: 58.68%
点击次数小于 10 的用户占比为: 81.67%
点击次数小于 20 的用户占比为: 94.28%
点击次数小于 30 的用户占比为: 97.66%
